In [ ]:
#generate test samples

In [ ]:
import cv2


In [24]:
import os
import cv2
TARGET_DIR = r"C:\Users\shris\OneDrive\Desktop\Face recogniser"
try:
    os.chdir(TARGET_DIR)
    print(f"Successfully changed:{os.getcwd()}")
except FileNotFoundError:
    print(f"Error:Directory not found at {TARGET_DIR}")
def generate_dataset():
    face_classifier = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3,5)
        #scaling factor = 1.3
        #Minimum neighbour = 5
        if len(faces) == 0:
             return None
        for (x,y,w,h) in faces :
             cropped_face=img[y:y+h, x:x+w]
        return cropped_face
    cap = cv2.VideoCapture(0)
    id=4
    img_id=1
    data_dir="data"
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    while True:
       ret,frame = cap.read()
       if face_cropped(frame) is not None :
            img_id+=1
            face = cv2.resize(face_cropped(frame), (200,200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            file_name_path = "data/user."+str(id)+"."+str(img_id)+".jpg"
            cv2.imwrite(file_name_path, face)
            cv2.putText(face, str(img_id), (50,50), cv2.FONT_HERSHEY_COMPLEX,1,(0,255,0),2)

            cv2.imshow("Cropped face", face)
            if cv2.waitKey(1)==13 or int(img_id)==200:
                break
    cap.release()
    cv2.destroyAllWindows()
    print("Collecting samples is complete")
generate_dataset()

Successfully changed:C:\Users\shris\OneDrive\Desktop\Face recogniser


In [18]:
import numpy as np
from PIL import Image
import os
import cv2
TARGET_DIR = r"C:\Users\shris\OneDrive\Desktop\Face recogniser" 

try:
    os.chdir(TARGET_DIR)
    print(f"Successfully changed working directory to: {os.getcwd()}")
    if 'data' in os.listdir('.'):
        print("data folder is now visible in the current directory.")
    else:
        print("⚠ Warning: 'data' folder not found after changing directory. Check your TARGET_DIR path.")

except FileNotFoundError:
    print(f"Error: Directory not found at {TARGET_DIR}. Please check the path.")
def train_classifier(data_dir):
    path = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
    faces = []
    ids = []
    for image in path:
        img= Image.open(image).convert('L');
        imageNp= np.array(img, 'uint8')
        id = int(os.path.split(image)[1].split(".")[1])

        faces.append(imageNp)
        ids.append(id)
    ids=np.array(ids)

    #train the classifier and save
    clf=cv2.face.LBPHFaceRecognizer_create()
    clf.train(faces, ids)
    clf.write("classifier.xml")
train_classifier("data")

Successfully changed working directory to: C:\Users\shris\OneDrive\Desktop\Face recogniser
data folder is now visible in the current directory.


In [28]:
import cv2
import numpy as np
from PIL import Image
import os
TARGET_DIR = r"C:\Users\shris\OneDrive\Desktop\Face recogniser" 
os.chdir(TARGET_DIR)

def draw_boundary(img, classifier, scaleFactor, minNeighbors, color, text, clf ):
    gray_image= cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = classifier.detectMultiScale(gray_image, scaleFactor, minNeighbors)
    coords = []

    for(x,y,w,h) in features :
        MIN_FACE_SIZE=50
        if w>=MIN_FACE_SIZE and h>=MIN_FACE_SIZE:
         cv2.rectangle(img, (x,y),(x+w, y+h), color, 2)
         id, pred = clf.predict(gray_image[y:y+h, x:x+w])
         confidence = int(100*(1-pred/300))

         if confidence>77:
            if id==1:
                cv2.putText(img, "Shristi", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
            elif (id==2):
                cv2.putText(img, "Nehuuuu", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
            elif (id==3):
                cv2.putText(img, "Gayuuuuuu", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
            elif (id==3):
                cv2.putText(img, "Dikshuuuuuu", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1, cv2.LINE_AA)
         else :
             cv2.putText(img, "Unknown", (x,y-5), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0, 0,255), 1, cv2.LINE_AA)
    
        coords.append([x,y,w,h])   
    return coords

def recognize(img, clf, faceCascade):
    coords = draw_boundary(img, faceCascade, 1.1, 10, (255,255,255), "Face", clf)
    return img
faceCascade=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
clf=cv2.face.LBPHFaceRecognizer_create()
clf.read("classifier.xml")
video_capture=cv2.VideoCapture(0)

while True:
    ret,img=video_capture.read()
    if ret == False or img is None:
        print("Error: Failed to read frame from video source. Breaking loop.")
        break
    img = recognize(img,clf,faceCascade)
    cv2.imshow("Face detection",img)

    if cv2.waitKey(1)==13:
        break

video_capture.release()
cv2.destroyAllWindows()